# 00 · Quickstart: your first Ragas evaluation in 10 minutes
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/ragas-evals-enterprise/blob/main/notebooks/00_quickstart.ipynb)

**Goal:** grade *one* answer from a real LLM application with three Ragas metrics, and see how a metric
catches a hallucination.

| | |
|---|---|
| ⏱️ Time | ~10 minutes |
| 💵 API cost | < $0.01 (gpt-4o-mini) |
| 🧰 Stack | LangChain 1.x (the app) · Ragas 0.4 (the evaluator) · OpenAI |

## The scenario: Aurora Bank 🏦
*Aurora Bank* (fictional) is rolling out a **customer-support assistant** that answers questions from
the bank's policy documents (cards, fees, wires, savings, privacy). It is a classic **RAG** system:
retrieve the relevant policy chunks, then let an LLM write the answer.

In a bank, a wrong answer is not just "a bit unhelpful":
* a hallucinated fee becomes a **customer complaint**;
* "you should buy Tesla stock" from a bank bot is a **regulatory incident**.

So before anything ships, we need to **measure** answer quality, automatically, repeatably, at scale.
That is what **evals** are, and **Ragas** is a library of ready-made, research-backed evaluation metrics.

### 🔑 Before you run anything
* **Colab:** click the 🔑 **Secrets** icon in the left sidebar → *Add new secret* → name `OPENAI_API_KEY`,
  paste your key, and switch on **Notebook access**. (If you skip this, the setup cell asks you to paste the key.)
* **Local:** copy `.env.example` to `.env` and put your key there. See the README for the full setup.

The setup cell clones the project, installs the pinned requirements (~1 min on Colab), and loads your key.
If Colab prints pip *dependency-resolver warnings* about other preinstalled packages, they are harmless.

In [1]:
# ⚙️ SETUP: run this cell first. It works in Google Colab and on your laptop.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/baluragala/ragas-evals-enterprise.git"  # ← change this if you forked the repo

if "google.colab" in sys.modules:                       # running in Colab: fetch the project + dependencies
    if not Path("/content/ragas-evals-enterprise").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ragas-evals-enterprise"], check=True)
    os.chdir("/content/ragas-evals-enterprise")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Find the project root (works from notebooks/ or the repo root) and make `aurora_evals` importable.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "aurora_evals").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["RAGAS_DO_NOT_TRACK"] = "true"   # turn off ragas telemetry (it also makes evals ~5x faster; see docs)

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)

from aurora_evals.config import setup_environment
setup_environment()   # finds OPENAI_API_KEY in your env / .env / Colab Secrets, or asks you to paste it
print("✅ Ready. Project root:", ROOT)

✅ Ready. Project root: /Users/balakrishna/Training/gen-ai/ragas-evals-enterprise


## Step 1: Ask the assistant a question

`AuroraRAG` (in `src/aurora_evals/rag_app.py`) is built with **LangChain 1.x**:

```
question ─► retriever (InMemoryVectorStore + OpenAI embeddings) ─► top-k policy chunks
                                                                     │
                     ChatPromptTemplate (grounding rules) │ ChatOpenAI ◄┘  ─► answer
```

It returns everything an evaluator needs: the **response**, the **retrieved contexts**, and telemetry
(latency, tokens).

> 💡 Notebooks support **top-level `await`**. LangChain and Ragas are async-first, so we `await` calls
> directly in cells. (In a `.py` script you would wrap them in `asyncio.run(...)`.)

In [2]:
from aurora_evals.rag_app import AuroraRAG

app = AuroraRAG("baseline")      # the configuration currently "in production"
question = "What is the foreign transaction fee on the Aurora Platinum card?"
result = await app.ainvoke(question)

print("ANSWER:  ", result.response)
print("SOURCES: ", result.retrieved_sources)
print(f"LATENCY:  {result.latency_s}s   TOKENS: {result.total_tokens}")

ANSWER:   The foreign transaction fee on the Aurora Platinum card is none (0%). [Sources: credit_cards.md]
SOURCES:  ['credit_cards.md', 'credit_cards.md', 'credit_cards.md', 'credit_cards.md']
LATENCY:  1.243s   TOKENS: 538


🔍 **Reading the output:** the answer should say Platinum has **no** (0%) foreign transaction fee and cite
`credit_cards.md`. Note that **several** chunks were retrieved; not all of them are relevant. That will matter
for *context precision* later.

## Step 2: Create the *judge*

Many Ragas metrics use an LLM as a **judge**: it reads the question, answer, and context and makes small,
structured decisions ("is this claim supported by this text? yes/no").

Ragas 0.4 builds judges with `llm_factory(model, client=...)`. Our helper `get_judge_llm()` wraps that with
**on-disk caching** (re-running an unchanged evaluation is free) and safe defaults (temperature 0, retries).

> Why not reuse the app's LangChain `ChatOpenAI`? Keeping the **judge separate from the system under test** is
> good practice: you can make the judge stronger than the app, and judge settings never leak into production.

In [3]:
from aurora_evals.judges import get_judge_llm, get_judge_embeddings

judge = get_judge_llm()                 # gpt-4o-mini by default (override with AURORA_JUDGE_MODEL)
print(judge)

InstructorLLM(provider='openai', model='gpt-4o-mini', client=<AsyncInstructor:async>, temperature=0, max_tokens=4096, top_p=0.1)


## Step 3: Score the answer with three metrics

| Metric | Question it answers | Inputs it needs | Uses a reference? |
|---|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? *(hallucination check)* | question, answer, contexts | ❌ no |
| **Context Recall** | Did retrieval find everything needed to state the *reference* answer? | question, contexts, reference | ✅ yes |
| **Factual Correctness** | How much does the answer's content overlap the reference's claims (F1)? | answer, reference | ✅ yes |

A **reference** is a human-written "gold" answer. Metrics that don't need one (*reference-free*) can also run
on live production traffic, which is very useful later.

In [4]:
from ragas.metrics.collections import Faithfulness, ContextRecall, FactualCorrectness

reference = "The Aurora Platinum card has no foreign transaction fee (0%)."

faithfulness = Faithfulness(llm=judge)
context_recall = ContextRecall(llm=judge)
factual_correctness = FactualCorrectness(llm=judge)

scores = {
    "faithfulness": await faithfulness.ascore(
        user_input=question, response=result.response, retrieved_contexts=result.retrieved_contexts),
    "context_recall": await context_recall.ascore(
        user_input=question, retrieved_contexts=result.retrieved_contexts, reference=reference),
    "factual_correctness": await factual_correctness.ascore(
        response=result.response, reference=reference),
}
pd.Series({name: r.value for name, r in scores.items()}, name="score").round(2)

faithfulness           1.0
context_recall         1.0
factual_correctness    1.0
Name: score, dtype: float64

🔍 **Reading the output:** all three are in **[0, 1]**, higher is better. For this easy question expect
faithfulness ≈ 1.0 and context recall ≈ 1.0. Factual correctness is often a bit lower (e.g. 0.67–1.0)
because it compares *claims* one by one: an answer that adds an extra true detail, or words things
differently, loses a little *precision* even when it is right. Treat it as a soft signal, not an exact grade.

## Step 4: Watch a metric catch a hallucination

✋ **Predict first:** we hand the metric an answer that *invents* two facts (a 1% fee and a $250 annual fee).
The context is unchanged. What will **faithfulness** be: close to 1, close to 0.5, or close to 0?

**How faithfulness works:**
1. The judge splits the answer into atomic **claims** ("the FX fee is 1%", "the annual fee is $250").
2. For each claim it checks: *can this be inferred from the retrieved context?*
3. `faithfulness = supported claims / total claims`

In [5]:
hallucinated = "The Aurora Platinum card charges a 1% foreign transaction fee and has a $250 annual fee."

bad = await faithfulness.ascore(
    user_input=question, response=hallucinated, retrieved_contexts=result.retrieved_contexts)
good = scores["faithfulness"]
print(f"faithfulness of the real answer:         {good.value:.2f}")
print(f"faithfulness of the hallucinated answer: {bad.value:.2f}")

faithfulness of the real answer:         1.00
faithfulness of the hallucinated answer: 0.00


🔍 **Reading the output:** the hallucinated answer should score **0.0**: neither claim is supported
(the context says 0% and $450). That is the core idea of LLM evaluation: **turn "is this answer good?"
into many small, checkable questions**, and let a judge answer them consistently.

## Step 5: Faithfulness is relative to the *context*, not to the truth

✋ **Predict first:** we keep the **correct** answer but pretend retrieval returned the *wrong* documents
(wire-transfer policy instead of credit cards). What happens to faithfulness and to context recall?

In [6]:
wrong_contexts = [
    "[Wire Transfers > Domestic wires] Daily limit in the app: $50,000. Fee: $25 per outgoing domestic wire.",
    "[Wire Transfers > International wires] Daily limit in the app: $25,000. Fee: $45 per outgoing wire.",
]
f = await faithfulness.ascore(user_input=question, response=result.response, retrieved_contexts=wrong_contexts)
r = await context_recall.ascore(user_input=question, retrieved_contexts=wrong_contexts, reference=reference)
print(f"faithfulness with wrong contexts:   {f.value:.2f}")
print(f"context recall with wrong contexts: {r.value:.2f}")

faithfulness with wrong contexts:   0.00
context recall with wrong contexts: 0.00


🔍 **Reading the output:** both should drop to (or near) **0**, even though the answer itself is *true*.

* **Context recall = 0** → retrieval failed to find the needed facts. That is a **retrieval** problem.
* **Faithfulness = 0** → the answer is *not supported by what was retrieved*. If retrieval truly returned this,
  the model must have used outside knowledge, which is exactly what a grounded bank assistant must not do.

This is the key diagnostic trick of RAG evaluation, used throughout the course:

| context recall | faithfulness | Likely root cause |
|---|---|---|
| low | — | **Retrieval**: fix chunking, embeddings, top-k, filters |
| high | low | **Generation**: the LLM hallucinated; fix the prompt or model |
| high | high, but answer still wrong | **Generation**: reasoning/arithmetic error, or a bad reference |

## ✅ Key takeaways
1. An **eval** = a dataset of inputs (+ optional references) **×** metrics **×** a judge.
2. **Faithfulness** catches hallucinations; **context recall** checks retrieval; **factual correctness** checks
   the answer against a human reference.
3. Metrics are **diagnostic**: the *pattern* across metrics tells you *where* the system broke.

➡️ **Next:** [`01_rag_evaluation.ipynb`](01_rag_evaluation.ipynb) scales this from 1 question to a full
**golden dataset** with a complete metric suite.